In [ ]:
!git clone https://github.com/deepseek-ai/Janus.git

Cloning into 'Janus'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 121 (delta 58), reused 39 (delta 39), pack-reused 37 (from 1)
Receiving objects: 100% (121/121), 7.18 MiB | 27.86 MiB/s, done.
Resolving deltas: 100% (59/59), done.


 "Janus Pro" :a publicly accessible collection of code and data for a multimodal AI model developed by DeepSeek. It is a novel autoregressive framework that unifies multimodal understanding and generation.

Navigate to the cloned directory, and install the required packages

In [ ]:
%cd Janus
!pip install -e .
!pip install flash-attn

/content/Janus
Obtaining file:///content/Janus
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for janus (pyproject.toml) ... done
  Created wheel for janus: filename=janus-1.0.0-0.editable-py3-none-any.whl size=15973 sha256=c2271b0d5bc34c6706643f8563fc7bb1dd47b86049ccb0268845fb1f1a16fd7a
  Stored in directory: /tmp/pip-ephem-wheel-cache-vbgscevd/wheels/b1/8b/89/d6c5737f8b7c6b17fee234aeeea7b70616b3a0f3e434fe0401
Successfully built janus
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [janus]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 37.0 MB/s  0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and

In [ ]:
#Load the model and move it to GPU
import torch
from transformers import AutoModelForCausalLM
from janus.models import MultiModalityCausalLM, VLChatProcessor
from janus.utils.io import load_pil_images

In [ ]:
model_path="deepseek-ai/Janus-Pro-1B"
vl_chat_processor=VLChatProcessor.from_pretrained(model_path)
tokenizer=vl_chat_processor.tokenizer

preprocessor_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.72M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/344 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


processor_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

In [ ]:
vl_gpt=AutoModelForCausalLM.from_pretrained(model_path,trust_remote_code=True)
vl_gpt=vl_gpt.to(torch.bfloat16).cuda().eval()

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
image_path="/content/drive/My Drive/DL/person_living_room.jpg"
#question="What's in the image?"
#question="Can you write an introduction scene for a novel based on this picture, assume the character name is sofia, who is a private detective with her dog boxer"
question="Can you identify objects in the image and list based on the category/type"

#question="Can you explain what you see in the image?"

In [ ]:
conversation=[{"role":"<|User|>","content":f"<image_placeholder>\n{question}","images":[image_path]},{"role":"<|Assistant|>","content":""}] #The string combines the image placeholder with the question text.
#Assistant's response and is initially empty. The model will fill this with its generated response.

In [ ]:
import textwrap

In [ ]:
#load image
pil_images=load_pil_images(conversation)
#prepare inputs for the model
prepare_inputs=vl_chat_processor(conversations=conversation,images=pil_images,force_batchify=True).to(vl_gpt.device) #ensures that the inputs are batchified (even if it's a single sample).
inputs_embeds=vl_gpt.prepare_inputs_embeds(**prepare_inputs) #prepare_inputs likely contains keys like input_ids, pixel_values, etc.
#generate response
outputs=vl_gpt.language_model.generate(inputs_embeds=inputs_embeds, #embedded multimodal input.
                                       attention_mask=prepare_inputs.attention_mask, #tells the model which tokens to attend to (and which to ignore).
                                       pad_token_id=tokenizer.eos_token_id, #special token for padding
                                       bos_token_id=tokenizer.bos_token_id, #special token for beginning of sequence
                                       eos_token_id=tokenizer.eos_token_id, #special token for end of sequence
                                       max_new_tokens=512, #limits how many tokens the model can generate.
                                       do_sample=False, #deterministic
                                       use_cache=True) #enables caching for speed during decoding.
#Decode and print response
#If the output tensor is on the GPU, this moves it to the CPU so it can be converted into a list (since .tolist() only works on CPU tensors).
answer=tokenizer.decode(outputs[0].cpu().tolist(),skip_special_tokens=True) #Translates the list of token IDs back into a human-readable string.

# Format and print the summary as a paragraph
wrapped_para = textwrap.fill(answer, width=125)
#print("DeepSeek Janus Response:\n", wrapped_para)
print(f"{prepare_inputs['sft_format'][0]}",wrapped_para)

You are a helpful language and vision assistant. You are able to understand the visual content that the user provides, and assist the user with a variety of tasks using natural language.

<|User|>: <image_placeholder>
Can you identify objects in the image and list based on the category/type

<|Assistant|>: Sure! Here are the identified objects in the image, categorized by type:  1. **Furniture:**    - A large brown leather chair.
- A smaller brown leather chair.    - A round white coffee table.    - A large brown bean bag chair.    - A white side table.
2. **Decorative Items:**    - A large black lamp with a white shade.    - A tall black vase with green foliage.    - A black
sculpture with white accents.    - A pink and white stacked object, possibly a cushion or decorative piece.    - A white lamp
with a white shade.  3. **Natural Elements:**    - A large window with a view of greenery outside.    - A potted plant on a
small white stand.    - A large green plant on a white stand.  4.